# CS336 GPU Kernels & Triton — T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/CS336_tmp/blob/main/cs336_gpu_kernels_triton_t4.ipynb)

Stanford CS336 Spring 2026 Lecture 5–6 + Assignment 2를 T4용으로 재구성했다.

반복 흐름: **PyTorch baseline → benchmark/profile → Triton → correctness → benchmark/profile → PTX → 병목/개선**

- Lecture 6: GeLU, Softmax, Row Sum, MatMul + ReLU
- Assignment 2: fused RMSNorm
- Lecture 5: FlashAttention-style forward

공식 자료:
- https://github.com/stanford-cs336/lectures
- https://github.com/stanford-cs336/assignment2-systems


In [ ]:
import importlib.util
import math
import subprocess
import sys

if importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])

import torch
import torch.nn.functional as F
import triton
import triton.language as tl
from torch.profiler import ProfilerActivity

assert torch.cuda.is_available(), "GPU runtime이 필요합니다."

device = "cuda"

print("PyTorch:", torch.__version__)
print("Triton :", triton.__version__)
print("GPU    :", torch.cuda.get_device_name())
print("CC     :", torch.cuda.get_device_capability())


In [ ]:
def bench(fn, warmup=10, trials=30):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    times = []
    for _ in range(trials):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        fn()
        end.record()

        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    return sum(times) / len(times)


def prof(fn, rows=10):
    for _ in range(3):
        fn()
    torch.cuda.synchronize()

    with torch.profiler.profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]
    ) as result:
        fn()
        torch.cuda.synchronize()

    print(
        result.key_averages().table(
            sort_by="cuda_time_total",
            row_limit=rows,
            max_name_column_width=90,
        )
    )


def ptx(handle, memory_only=False, n=80):
    lines = handle.asm["ptx"].splitlines()

    if memory_only:
        keys = ("ld.global", "st.global", "%ctaid", "%tid", ".reg")
        lines = [line for line in lines if any(key in line for key in keys)]

    print("\n".join(lines[:n]))


def compare(implementations):
    for name, fn in implementations.items():
        try:
            print(f"{name:28s} {bench(fn):8.4f} ms")
        except Exception as error:
            print(f"{name:28s} ERROR: {error}")


## 1. GeLU — elementwise / fusion / 첫 PTX


In [ ]:
def gelu_naive(x):
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))


try:
    gelu_compiled = torch.compile(gelu_naive)
except Exception:
    gelu_compiled = None


@triton.jit
def gelu_kernel(x_ptr, y_ptr, num_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    program_id = tl.program_id(axis=0)
    offsets = program_id * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < num_elements

    x = tl.load(x_ptr + offsets, mask=mask)
    y = 0.5 * x * (1.0 + tl.erf(x * 0.7071067811865476))

    tl.store(y_ptr + offsets, y, mask=mask)


def gelu_triton(x, return_handle=False):
    y = torch.empty_like(x)

    block_size = 256
    grid = (triton.cdiv(x.numel(), block_size),)

    handle = gelu_kernel[grid](
        x,
        y,
        x.numel(),
        BLOCK_SIZE=block_size,
    )

    return (y, handle) if return_handle else y


x = torch.randn(4_000_000, device=device)

y, gelu_handle = gelu_triton(x, return_handle=True)
torch.testing.assert_close(y, gelu_naive(x), rtol=1e-4, atol=1e-5)

print("naive profile")
prof(lambda: gelu_naive(x))

print("builtin profile")
prof(lambda: F.gelu(x))

compare(
    {
        "naive": lambda: gelu_naive(x),
        "builtin": lambda: F.gelu(x),
        "triton": lambda: gelu_triton(x),
    }
)

if gelu_compiled is not None:
    try:
        gelu_compiled(x)
        compare({"torch.compile": lambda: gelu_compiled(x)})
    except Exception as error:
        print("compile skipped:", error)


In [ ]:
# 첫 예제에서는 PTX를 비교적 넓게 본다.
ptx(gelu_handle, n=100)


**확인:** naive의 여러 primitive kernel/HBM 왕복과 fused kernel을 비교한다.  
첫 PTX에서는 global load/store, block/thread, register를 직접 본다.


## 2. Softmax — row reduction + fusion


In [ ]:
def softmax_naive(x):
    row_max = x.max(dim=-1, keepdim=True).values
    shifted = x - row_max
    exp_x = torch.exp(shifted)
    row_sum = exp_x.sum(dim=-1, keepdim=True)

    return exp_x / row_sum


@triton.jit
def softmax_kernel(x_ptr, y_ptr, num_columns: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(axis=0)
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < num_columns

    x = tl.load(
        x_ptr + row * num_columns + offsets,
        mask=mask,
        other=-float("inf"),
    )

    x = x - tl.max(x, axis=0)
    numerator = tl.exp(x)
    denominator = tl.sum(numerator, axis=0)
    output = numerator / denominator

    tl.store(
        y_ptr + row * num_columns + offsets,
        output,
        mask=mask,
    )


def softmax_triton(x, return_handle=False):
    num_rows, num_columns = x.shape
    block_size = triton.next_power_of_2(num_columns)
    num_warps = 8 if block_size >= 2048 else 4

    y = torch.empty_like(x)

    handle = softmax_kernel[(num_rows,)](
        x,
        y,
        num_columns,
        BLOCK_SIZE=block_size,
        num_warps=num_warps,
    )

    return (y, handle) if return_handle else y


x_softmax = torch.randn(4096, 1024, device=device)

y_softmax, softmax_handle = softmax_triton(x_softmax, return_handle=True)

torch.testing.assert_close(
    y_softmax,
    torch.softmax(x_softmax, dim=-1),
    rtol=2e-4,
    atol=2e-5,
)

print("naive")
prof(lambda: softmax_naive(x_softmax))

print("triton")
prof(lambda: softmax_triton(x_softmax))

compare(
    {
        "naive": lambda: softmax_naive(x_softmax),
        "builtin": lambda: torch.softmax(x_softmax, dim=-1),
        "triton": lambda: softmax_triton(x_softmax),
    }
)

ptx(softmax_handle, memory_only=True)


**확인:** `max → exp → sum → divide`가 여러 kernel인지 확인한다.  
한 row를 하나의 Triton program에서 처리할 때 어떤 HBM 왕복이 사라지는지 본다.


## 3. Row Sum — baby tiling


In [ ]:
@triton.jit
def row_sum_kernel(x_ptr, y_ptr, num_columns: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(axis=0)
    accumulator = tl.zeros((BLOCK_SIZE,), dtype=tl.float32)

    for start in range(0, num_columns, BLOCK_SIZE):
        offsets = start + tl.arange(0, BLOCK_SIZE)

        values = tl.load(
            x_ptr + row * num_columns + offsets,
            mask=offsets < num_columns,
            other=0.0,
        )

        accumulator += values

    row_sum = tl.sum(accumulator, axis=0)
    tl.store(y_ptr + row, row_sum)


def row_sum_triton(x, block_size=1024, return_handle=False):
    num_rows, num_columns = x.shape

    y = torch.empty(
        num_rows,
        device=x.device,
        dtype=torch.float32,
    )

    handle = row_sum_kernel[(num_rows,)](
        x,
        y,
        num_columns,
        BLOCK_SIZE=block_size,
        num_warps=8,
    )

    return (y, handle) if return_handle else y


x_row_sum = torch.randn(1024, 16384, device=device)

y_row_sum, row_sum_handle = row_sum_triton(
    x_row_sum,
    return_handle=True,
)

torch.testing.assert_close(
    y_row_sum,
    x_row_sum.sum(dim=-1),
    rtol=2e-4,
    atol=2e-3,
)

for block_size in [256, 512, 1024, 2048]:
    latency = bench(
        lambda block_size=block_size: row_sum_triton(
            x_row_sum,
            block_size=block_size,
        )
    )
    print(f"BLOCK_SIZE={block_size:4d}: {latency:.4f} ms")

compare(
    {
        "torch.sum": lambda: x_row_sum.sum(dim=-1),
        "triton": lambda: row_sum_triton(x_row_sum),
    }
)

ptx(row_sum_handle, memory_only=True)


**확인:** row가 한 block에 들어가지 않으면 tile을 순회한다.  
`BLOCK_SIZE` sweep으로 tile 크기와 latency 변화를 확인한다.


## 4. MatMul + ReLU — tiling / compute / epilogue fusion


In [ ]:
@triton.jit
def matmul_relu_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    stride_am: tl.constexpr,
    stride_ak: tl.constexpr,
    stride_bk: tl.constexpr,
    stride_bn: tl.constexpr,
    stride_cm: tl.constexpr,
    stride_cn: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    program_m = tl.program_id(axis=0)
    program_n = tl.program_id(axis=1)

    offsets_m = program_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offsets_n = program_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offsets_k = tl.arange(0, BLOCK_K)

    a_ptrs = (
        a_ptr
        + offsets_m[:, None] * stride_am
        + offsets_k[None, :] * stride_ak
    )

    b_ptrs = (
        b_ptr
        + offsets_k[:, None] * stride_bk
        + offsets_n[None, :] * stride_bn
    )

    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k_start in range(0, K, BLOCK_K):
        a_tile = tl.load(
            a_ptrs,
            mask=(offsets_m[:, None] < M) & (k_start + offsets_k[None, :] < K),
            other=0.0,
        )

        b_tile = tl.load(
            b_ptrs,
            mask=(k_start + offsets_k[:, None] < K) & (offsets_n[None, :] < N),
            other=0.0,
        )

        accumulator += tl.dot(a_tile, b_tile)

        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # epilogue fusion: ReLU를 별도 kernel로 실행하지 않는다.
    accumulator = tl.maximum(accumulator, 0.0)

    c_ptrs = (
        c_ptr
        + offsets_m[:, None] * stride_cm
        + offsets_n[None, :] * stride_cn
    )

    output_mask = (offsets_m[:, None] < M) & (offsets_n[None, :] < N)

    tl.store(c_ptrs, accumulator, mask=output_mask)


def matmul_relu_triton(
    a,
    b,
    block_m=32,
    block_n=32,
    block_k=32,
    return_handle=False,
):
    M, K = a.shape
    N = b.shape[1]

    c = torch.empty(
        (M, N),
        device=a.device,
        dtype=torch.float32,
    )

    grid = (
        triton.cdiv(M, block_m),
        triton.cdiv(N, block_n),
    )

    handle = matmul_relu_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        *a.stride(),
        *b.stride(),
        *c.stride(),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
        num_warps=4,
    )

    return (c, handle) if return_handle else c


a = torch.randn(1024, 1024, device=device, dtype=torch.float16)
b = torch.randn(1024, 1024, device=device, dtype=torch.float16)

c, matmul_handle = matmul_relu_triton(a, b, return_handle=True)

torch.testing.assert_close(
    c,
    torch.relu(a @ b).float(),
    rtol=2e-2,
    atol=2e-1,
)

compare(
    {
        "torch matmul + relu": lambda: torch.relu(a @ b),
        "triton tiled + relu": lambda: matmul_relu_triton(a, b),
    }
)

configs = [
    (16, 16, 32),
    (32, 32, 32),
    (64, 32, 32),
]

for block_m, block_n, block_k in configs:
    latency = bench(
        lambda block_m=block_m,
               block_n=block_n,
               block_k=block_k: matmul_relu_triton(
            a,
            b,
            block_m=block_m,
            block_n=block_n,
            block_k=block_k,
        )
    )

    print(
        f"BM={block_m:2d}, "
        f"BN={block_n:2d}, "
        f"BK={block_k:2d} "
        f"-> {latency:.4f} ms"
    )

ptx(matmul_handle, memory_only=True)


**확인:** tiling은 A/B tile을 재사용해 arithmetic intensity를 높인다.  
ReLU를 output write 전에 붙이면 별도 ReLU kernel과 HBM 왕복을 없앨 수 있다.


## 5. RMSNorm — Assignment 2 style fused memory-bound kernel


In [ ]:
def rms_norm_naive(x, weight, eps=1e-6):
    mean_square = x.float().pow(2).mean(dim=-1, keepdim=True)
    inverse_rms = torch.rsqrt(mean_square + eps)

    normalized = x.float() * inverse_rms * weight.float()
    return normalized.to(dtype=x.dtype)


@triton.jit
def rms_norm_kernel(
    x_ptr,
    weight_ptr,
    y_ptr,
    num_columns: tl.constexpr,
    eps: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(axis=0)
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < num_columns

    x = tl.load(
        x_ptr + row * num_columns + offsets,
        mask=mask,
        other=0.0,
    ).to(tl.float32)

    weight = tl.load(
        weight_ptr + offsets,
        mask=mask,
        other=0.0,
    ).to(tl.float32)

    mean_square = tl.sum(x * x, axis=0) / num_columns
    inverse_rms = tl.rsqrt(mean_square + eps)
    output = x * inverse_rms * weight

    tl.store(
        y_ptr + row * num_columns + offsets,
        output,
        mask=mask,
    )


def rms_norm_triton(x, weight, eps=1e-6, return_handle=False):
    num_rows, num_columns = x.shape

    block_size = triton.next_power_of_2(num_columns)
    num_warps = 8 if block_size >= 2048 else 4

    y = torch.empty_like(x)

    handle = rms_norm_kernel[(num_rows,)](
        x,
        weight,
        y,
        num_columns,
        eps,
        BLOCK_SIZE=block_size,
        num_warps=num_warps,
    )

    return (y, handle) if return_handle else y


x_rms = torch.randn(4096, 1024, device=device, dtype=torch.float16)
weight_rms = torch.randn(1024, device=device, dtype=torch.float16)

y_rms, rms_handle = rms_norm_triton(
    x_rms,
    weight_rms,
    return_handle=True,
)

torch.testing.assert_close(
    y_rms,
    rms_norm_naive(x_rms, weight_rms),
    rtol=3e-3,
    atol=3e-3,
)

print("naive")
prof(lambda: rms_norm_naive(x_rms, weight_rms))

print("triton")
prof(lambda: rms_norm_triton(x_rms, weight_rms))

compare(
    {
        "naive RMSNorm": lambda: rms_norm_naive(x_rms, weight_rms),
        "triton fused": lambda: rms_norm_triton(x_rms, weight_rms),
    }
)

ptx(rms_handle, memory_only=True)


**확인:** RMSNorm은 FLOPs보다 memory movement가 중요할 수 있다.  
primitive kernel 수와 fused kernel 수를 profiler에서 비교한다.


## 6. FlashAttention-style forward — tiled matmul + online softmax


In [ ]:
def attention_naive(q, k, v):
    scale = 1.0 / math.sqrt(q.shape[-1])
    scores = (q @ k.transpose(-2, -1)) * scale
    probabilities = torch.softmax(scores, dim=-1)

    return probabilities @ v


@triton.jit
def flash_attention_forward_kernel(
    q_ptr,
    k_ptr,
    v_ptr,
    out_ptr,
    stride_qb: tl.constexpr,
    stride_qh: tl.constexpr,
    stride_qn: tl.constexpr,
    stride_qd: tl.constexpr,
    stride_kb: tl.constexpr,
    stride_kh: tl.constexpr,
    stride_kn: tl.constexpr,
    stride_kd: tl.constexpr,
    stride_vb: tl.constexpr,
    stride_vh: tl.constexpr,
    stride_vn: tl.constexpr,
    stride_vd: tl.constexpr,
    stride_ob: tl.constexpr,
    stride_oh: tl.constexpr,
    stride_on: tl.constexpr,
    stride_od: tl.constexpr,
    num_heads: tl.constexpr,
    seq_len: tl.constexpr,
    head_dim: tl.constexpr,
    scale: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    query_block_id = tl.program_id(axis=0)
    batch_head_id = tl.program_id(axis=1)

    batch_id = batch_head_id // num_heads
    head_id = batch_head_id % num_heads

    query_offsets = query_block_id * BLOCK_M + tl.arange(0, BLOCK_M)
    key_offsets_local = tl.arange(0, BLOCK_N)
    dim_offsets = tl.arange(0, head_dim)

    q_ptrs = (
        q_ptr
        + batch_id * stride_qb
        + head_id * stride_qh
        + query_offsets[:, None] * stride_qn
        + dim_offsets[None, :] * stride_qd
    )

    q_block = tl.load(
        q_ptrs,
        mask=(query_offsets[:, None] < seq_len) & (dim_offsets[None, :] < head_dim),
        other=0.0,
    )

    running_max = tl.full((BLOCK_M,), -float("inf"), dtype=tl.float32)
    running_sum = tl.zeros((BLOCK_M,), dtype=tl.float32)
    accumulator = tl.zeros((BLOCK_M, head_dim), dtype=tl.float32)

    for key_start in range(0, seq_len, BLOCK_N):
        key_offsets = key_start + key_offsets_local

        k_ptrs = (
            k_ptr
            + batch_id * stride_kb
            + head_id * stride_kh
            + key_offsets[:, None] * stride_kn
            + dim_offsets[None, :] * stride_kd
        )

        v_ptrs = (
            v_ptr
            + batch_id * stride_vb
            + head_id * stride_vh
            + key_offsets[:, None] * stride_vn
            + dim_offsets[None, :] * stride_vd
        )

        kv_mask = (key_offsets[:, None] < seq_len) & (dim_offsets[None, :] < head_dim)

        k_block = tl.load(k_ptrs, mask=kv_mask, other=0.0)
        v_block = tl.load(v_ptrs, mask=kv_mask, other=0.0)

        qk = tl.dot(q_block, tl.trans(k_block)) * scale

        qk = tl.where(
            (query_offsets[:, None] < seq_len) & (key_offsets[None, :] < seq_len),
            qk,
            -float("inf"),
        )

        block_max = tl.max(qk, axis=1)
        new_max = tl.maximum(running_max, block_max)
        correction = tl.exp(running_max - new_max)

        probabilities = tl.exp(qk - new_max[:, None])
        block_sum = tl.sum(probabilities, axis=1)

        accumulator = (
            accumulator * correction[:, None]
            + tl.dot(probabilities.to(tl.float16), v_block)
        )

        running_sum = running_sum * correction + block_sum
        running_max = new_max

    output = accumulator / running_sum[:, None]

    out_ptrs = (
        out_ptr
        + batch_id * stride_ob
        + head_id * stride_oh
        + query_offsets[:, None] * stride_on
        + dim_offsets[None, :] * stride_od
    )

    tl.store(
        out_ptrs,
        output,
        mask=(query_offsets[:, None] < seq_len) & (dim_offsets[None, :] < head_dim),
    )


def flash_attention_triton(
    q,
    k,
    v,
    block_m=32,
    block_n=32,
    return_handle=False,
):
    batch_size, num_heads, seq_len, head_dim = q.shape
    output = torch.empty_like(q)

    grid = (
        triton.cdiv(seq_len, block_m),
        batch_size * num_heads,
    )

    handle = flash_attention_forward_kernel[grid](
        q,
        k,
        v,
        output,
        *q.stride(),
        *k.stride(),
        *v.stride(),
        *output.stride(),
        num_heads,
        seq_len,
        head_dim,
        1.0 / math.sqrt(head_dim),
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        num_warps=4,
    )

    return (output, handle) if return_handle else output


q = torch.randn(1, 4, 256, 64, device=device, dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)

output, flash_handle = flash_attention_triton(
    q,
    k,
    v,
    return_handle=True,
)

torch.testing.assert_close(
    output,
    attention_naive(q, k, v),
    rtol=2e-2,
    atol=2e-2,
)

print("naive")
prof(lambda: attention_naive(q, k, v))

print("flash-style")
prof(lambda: flash_attention_triton(q, k, v))

compare(
    {
        "naive attention": lambda: attention_naive(q, k, v),
        "triton flash-style": lambda: flash_attention_triton(q, k, v),
    }
)

ptx(flash_handle, memory_only=True)


**확인:** naive는 \(QK^\top\) score와 softmax probability를 HBM에 materialize한다.  
FlashAttention-style은 Q tile을 잡고 K/V tile을 순회하면서 online softmax를 갱신해 전체 \(N\times N\) 중간 tensor를 쓰지 않는다.

## 반복 체크표

| 연산 | 핵심 |
|---|---|
| GeLU | elementwise + fusion + PTX |
| Softmax | reduction + fusion |
| Row Sum | baby tiling |
| MatMul+ReLU | tiled reuse + epilogue fusion |
| RMSNorm | memory-bound fused kernel |
| Attention | tiled matmul + online softmax + HBM traffic 제거 |

각 항목에서 마지막 질문은 동일하다:

**실제로 어떤 GPU kernel이 실행됐고, 병목은 무엇이었으며, custom kernel이 무엇을 줄였는가?**
